In [ ]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

from src.datasets import FaceDataset, train_transform, val_transform, FaceTestDataset
from src.model import MyCNN
from src.engine import train_one_epoch, evaluate
from src.utils import find_best_threshold

In [8]:
DATA_DIR = Path("dataset")
TRAIN_DIR = DATA_DIR / "train_images"
TRAIN_CSV = DATA_DIR / "train_solution.csv"
TEST_DIR = DATA_DIR / "test_images"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda
NVIDIA GeForce RTX 3080


In [3]:
df = pd.read_csv(TRAIN_CSV, header=None, names=["Id", "target"])
df["path"] = df["Id"].apply(lambda x: TRAIN_DIR / f"{x}.jpg")

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)

train_df = train_df.sample(min(1000, len(train_df)), random_state=42)
val_df = val_df.sample(min(200, len(val_df)), random_state=42)

print(len(train_df), len(val_df))

1000 200


In [ ]:
train_dataset = FaceDataset(train_df, transform=train_transform)
val_dataset = FaceDataset(val_df, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

In [5]:
model = MyCNN().to(device)

num_pos = (train_df["target"] == 1).sum()
num_neg = (train_df["target"] == 0).sum()
pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [6]:
import copy

best_val_f1 = -1
best_threshold = 0.5
best_state = None

n_epochs = 10

for epoch in range(n_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_f1_05, val_probs, val_labels = evaluate(
        model, val_loader, criterion, device, threshold=0.5
    )

    thr, val_f1_best = find_best_threshold(val_labels, val_probs)

    print(
        f"Epoch {epoch+1}/{n_epochs} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1@0.5={val_f1_05:.4f} | "
        f"best_thr={thr:.2f} | "
        f"best_val_f1={val_f1_best:.4f}"
    )

    if val_f1_best > best_val_f1:
        best_val_f1 = val_f1_best
        best_threshold = thr
        best_state = copy.deepcopy(model.state_dict())

    scheduler.step(val_f1_best)

Epoch 1/10 | train_loss=1.4706 | val_loss=1.5645 | val_f1@0.5=0.3596 | best_thr=0.37 | best_val_f1=0.4370
Epoch 2/10 | train_loss=1.2362 | val_loss=1.5257 | val_f1@0.5=0.2769 | best_thr=0.38 | best_val_f1=0.4706
Epoch 3/10 | train_loss=1.2151 | val_loss=1.7736 | val_f1@0.5=0.2295 | best_thr=0.18 | best_val_f1=0.4706
Epoch 4/10 | train_loss=1.1587 | val_loss=1.1785 | val_f1@0.5=0.4458 | best_thr=0.53 | best_val_f1=0.4636
Epoch 5/10 | train_loss=1.1020 | val_loss=1.2161 | val_f1@0.5=0.4812 | best_thr=0.50 | best_val_f1=0.4812
Epoch 6/10 | train_loss=1.0947 | val_loss=1.3032 | val_f1@0.5=0.3838 | best_thr=0.35 | best_val_f1=0.4690
Epoch 7/10 | train_loss=1.1347 | val_loss=1.4856 | val_f1@0.5=0.3467 | best_thr=0.27 | best_val_f1=0.4667
Epoch 8/10 | train_loss=1.1023 | val_loss=1.4412 | val_f1@0.5=0.3448 | best_thr=0.38 | best_val_f1=0.4833
Epoch 9/10 | train_loss=1.0666 | val_loss=1.1491 | val_f1@0.5=0.4308 | best_thr=0.65 | best_val_f1=0.5211
Epoch 10/10 | train_loss=1.0790 | val_loss=1.1

In [7]:
model.load_state_dict(best_state)
print("Best F1:", best_val_f1)
print("Best threshold:", best_threshold)

Best F1: 0.5211267605633803
Best threshold: 0.6500000000000001


In [12]:
from torchvision import transforms
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    ),
])

In [ ]:
test_dataset = FaceTestDataset(TEST_DIR, transform=test_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print("Test size:", len(test_dataset))

Test size: 10000


In [17]:
model.load_state_dict(best_state)
model.eval()

print("Best threshold:", best_threshold)

Best threshold: 0.6500000000000001


In [20]:
from tqdm import tqdm

@torch.no_grad()
def predict_test(model, loader, device, threshold):
    model.eval()

    ids = []
    preds = []

    for images, image_ids in tqdm(loader, desc="Predicting"):
        images = images.to(device)

        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        batch_preds = (probs >= threshold).astype(int)

        ids.extend(image_ids.numpy().tolist())
        preds.extend(batch_preds.tolist())

    return ids, preds

In [21]:
test_ids, test_preds = predict_test(
    model=model,
    loader=test_loader,
    device=device,
    threshold=best_threshold
)

print(test_ids[:10])
print(test_preds[:10])
print(len(test_ids), len(test_preds))

Predicting: 100%|██████████| 313/313 [01:56<00:00,  2.69it/s]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[1, 1, 1, 1, 0, 0, 0, 0, 1, 0]
10000 10000


In [25]:
submission = pd.DataFrame({
    "id": test_ids,
    "target_feature": test_preds
}).sort_values("id").reset_index(drop=True)

submission.head()
submission.to_csv("submission.csv", index=False)
print("submission.csv saved")

submission.csv saved
